# 13 · E-MNIST: espectro DMAP para varios β y experimento reproducible

Paquete experimental: encoder espectral congelado → dos cabezas afines → VAE, ahora sobre **E-MNIST** (split `balanced`: 47 clases de dígitos y letras, 28×28).

1. **Espectro y dimensión de Weyl.** Autovalores λ_k del operador DMAP congelado, k = 1…k (**k = 1000**), para varios β = factor × β_base (heurística de la mediana sobre las entradas estandarizadas de entrenamiento, como `experiment.py`), sobre **todas** las imágenes de entrenamiento muestreadas (N en `spectrum.npz`; la matriz densa no cabe en memoria). Se resuelve **sin formar la matriz**: Krylov por bloques con reortogonalización completa + Rayleigh–Ritz sobre el mismo operador que `SpectralMap.fit`, con el kernel por bloques de PyTorch (fp32 en GPU): FlashDiffusion solo aplica un vector por pasada y, a N completo, costaría horas por β. El experimento (sección 2) sigue usando `flash`. Los residuos de Ritz se guardan y solo cuentan los modos con residuo < `tol`. La ley de Weyl, λ_i ~ i^(2/d) con λ_i = −ln μ_i, da la dimensión d = 2 / pendiente de ln λ_i contra ln i en la ventana `weyl_window`. `checks.py` verifica Krylov contra el camino denso y la estimación de d sobre toros planos.
2. **Experimento.** El mismo pipeline piloto que `03_mnist_kaggle` (ae, vae, direct, plainvae; modos 2, 8, 32; tres semillas, una por proceso en las dos T4).

**Datos:** adjunta el Kaggle Dataset **crawford/emnist** (el workflow de GitHub lo adjunta solo). Sus CSV guardan cada imagen **transpuesta**; `data.ensure_emnist` la endereza y la guarda en caché como NPZ estilo MNIST. E-MNIST no tiene una carta conocida: como en MNIST, la exactitud kNN de la clase es solo una sonda, no recuperación de carta.

Selecciona **GPU T4 x2**. El backend `flash` compila el kernel upstream incluido para SM75.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json', 'spectrum.py',
        'config_emnist.json', 'config_spectrum_emnist.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## Datos E-MNIST

Se prepara la caché **una sola vez** antes de lanzar procesos en paralelo (la escritura es atómica de todos modos).

In [ ]:
from data import ensure_emnist
EMNIST = ensure_emnist('/kaggle/working/emnist_balanced.npz', 'balanced')
import numpy as np
with np.load(EMNIST) as a:
    print({k: a[k].shape for k in a.files}, 'clases:', int(a['y_train'].max()) + 1)

## Espectro de autovalores y dimensión de Weyl para varios β

N anclas = todas las imágenes de entrenamiento muestreadas (el valor exacto está en `spectrum.npz` y en la figura). Panel (a): λ_k contra k en escala log-log, una curva por β, con la dimensión de Weyl d en la leyenda. Panel (b): los mismos valores como un "espectrograma" β × k en log₁₀. Panel (c): dimensión local d(i) ajustada en ventanas [i, 2i]; la ventana sombreada es la de d. λ_0 = 1 (Perron) se omite. Los valores por debajo de 1e-16 son redondeo y se recortan solo en la figura; el NPZ guarda los valores completos. Los β grandes quedan fuera del régimen de Weyl (kernel más estrecho que el espaciado entre vecinos).

In [ ]:
spec_cfg = json.loads((ROOT / 'config_spectrum_emnist.json').read_text())
spec_cfg['data'] = str(EMNIST)
SPEC_CONFIG = ROOT / 'active_spectrum.json'
SPEC_CONFIG.write_text(json.dumps(spec_cfg, indent=2))
subprocess.run([sys.executable, str(ROOT / 'spectrum.py'), str(SPEC_CONFIG)], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'))
from IPython.display import display, Image
display(Image(filename=str(Path(spec_cfg['out']) / 'spectrum.png')))

In [ ]:
meta = json.loads((Path(spec_cfg['out']) / 'spectrum_meta.json').read_text())
print(f"N = {meta['n_anchors']}, método {meta['method']}, backend {meta['backend']}")
print(f"{'factor':>8} {'beta':>10} {'Weyl d':>8} {'max residuo (ventana)':>22}")
for r in meta['weyl']:
    print(f"{r['factor']:>8g} {r['beta']:>10.4g} {r['weyl_d']:>8.2f} {r['max_window_residual']:>22.1e}")

## Experimento: configuración

Mantiene el test oficial separado. El subconjunto se selecciona con semilla fija.

In [ ]:
cfg = json.loads((ROOT / 'config_emnist.json').read_text())
# PILOTO: verifica que el pipeline funciona. No es una réplica cuantitativa del artículo.
PAPER_BUDGET = False
if PAPER_BUDGET:
    cfg.update(n=12000, modes=[2,4,8,16,32,64,128,256], epochs=200)
# beta=None usa la heurística de mediana de entrenamiento * beta_factor, NO beta-flow.
cfg['backend'] = 'flash'   # cambiar explícitamente a 'torch' si se desea el backend alternativo
cfg['seeds'] = [0, 1, 2]
cfg['data'] = str(EMNIST)
cfg['out'] = '/kaggle/working/results_emnist'
CONFIG = ROOT / 'active_config.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

## Ejecutar en dos GPUs

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'launch.py'), str(CONFIG)], check=True)

## Resultados

MSE de reconstrucción fuera de muestra y `knn_label_accuracy` (sonda de clases, no carta). Barras: desviación entre semillas, no intervalos de confianza. Selecciona configuración con `validation_mse`; reserva test para la evaluación final.

In [ ]:
import pandas as pd
RESULTS = Path(cfg['out'])
display(pd.read_csv(RESULTS / 'metrics.csv'))
for figure in sorted(RESULTS.glob('*.png')):
    display(Image(filename=str(figure)))

## Archivos guardados

`spectrum_emnist/`: `spectrum.npz` (autovalores, residuos, d de Weyl, factores, β), `spectrum.png/.pdf`, `spectrum_meta.json`. `results_emnist/`: lo mismo que el notebook 03 (encoder, pesos, métricas, figuras).